In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/prompt-engineering-math/sample_submission.csv
/kaggle/input/prompt-engineering-math/test_with_translation.csv
/kaggle/input/prompt-engineering-math/train.csv
/kaggle/input/prompt-engineering-math/test.csv


In [ ]:
# Kaggle Prompt Engineering for Math - Groq + Llama-4-Maverick
# Final working version - December 2025

import os
import re
import pandas as pd
from groq import Groq


In [ ]:

# === CONFIGURATION ===
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")  # Set in Kaggle Secrets or env
if not GROQ_API_KEY:
    raise ValueError("Please set GROQ_API_KEY in your environment (Kaggle Secrets recommended)")

MODEL = "llama-4-maverick"  # Fastest & strongest math model on Groq right now


In [ ]:
# === BEST PROMPT FOR MATH (tuned for Llama-4-Maverick) ===
SYSTEM_PROMPT = """
You are a world-class mathematician. Solve the given problem using careful step-by-step reasoning (chain of thought).
Think slowly and clearly. After you finish reasoning, put the final answer inside <answer></answer> tags exactly as shown.

Rules:
- The answer must be a single number (integer or decimal).
- Never write units, words, or explanations after the closing tag.
- Use a dot as decimal separator (e.g., 3.14, not 3,14).
- Do not add trailing zeros (12.0 → 12, 5.500 → 5.5).

Examples:
<answer>42</answer>
<answer>-0.75</answer>
<answer>100</answer>
"""


In [ ]:


# === API WRAPPER ===
def get_answer_from_groq(problem_text: str) -> str:
    client = Groq(api_key=GROQ_API_KEY)
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": problem_text}
            ],
            temperature=0.0,
            max_tokens=1024,
            top_p=1.0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Groq API error: {e}")
        return ""


In [ ]:

# === ROBUST ANSWER EXTRACTION & FORMATTING ===
def extract_and_clean_answer(raw_response: str) -> str:
    if not raw_response:
        return "0"
    
    # Find content inside <answer>...</answer>
    match = re.search(r'<answer>(.*?)</answer>', raw_response, re.DOTALL | re.IGNORECASE)
    if not match:
        return "0"
    
    num_str = match.group(1).strip()
    
    # Clean possible junk
    num_str = num_str.replace(' ', '').replace(',', '.')
    
    try:
        num = float(num_str)
        if num.is_integer():
            return str(int(num))
        else:
            # Remove trailing zeros and dot
            return str(num).rstrip('0').rstrip('.')
    except ValueError:
        return "0"


In [ ]:

# === MAIN EXECUTION ===
def main():
    main():
    # Kaggle file paths
    test_path = "/kaggle/input/prompt-engineering-math/test_with_translation.csv"
    sample_sub_path = "/kaggle/input/prompt-engineering-math/sample_submission.csv"
    output_path = "/kaggle/working/submission.csv"

    print("Loading test data...")
    df = pd.read_csv(test_path)

    results = []
    for idx, row in df.iterrows():
        problem_id = row['problem_id']
        problem_en = row['translation']  # English version
        
        print(f"Processing {problem_id} ({idx+1}/{len(df)})...")
        
        raw = get_answer = get_answer_from_groq(problem_en)
        clean_answer = extract_and_clean_answer(raw_answer)
        
        results.append({"id": problem_id, "answer": clean_answer})

    # Save submission
    submission = pd.DataFrame(results)
    submission.to_csv(output_path, index=False)
    print(f"\nSubmission saved to:", output_path)
    display(submission.head(10))

if __name__ == "__main__":
    main()